# Facial Emotion Recognition - Mini Project

This notebook demonstrates how to build and train a CNN model for facial emotion recognition.

## Project Overview
- **Goal**: Build a facial emotion recognition model using CNN
- **Data**: Photos of you and your friend with different emotions
- **Purpose**: Learn CNN and deep learning fundamentals

## 1. Import Libraries

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.models import load_model

# Add project modules to path
sys.path.append('..')

from models.cnn_model import create_simple_cnn, create_basic_cnn, compile_model, print_model_summary
from utils.preprocessing import FaceDetector, preprocess_image, prepare_dataset_from_directory
from predict import EmotionPredictor

print("Libraries imported successfully!")

## 2. Understanding the Data Structure

Before training, organize your images in the following structure:

```
data/
  train/
    happy/
      person1_happy_1.jpg
      person2_happy_1.jpg
    sad/
      person1_sad_1.jpg
      person2_sad_1.jpg
    neutral/
      person1_neutral_1.jpg
      person2_neutral_1.jpg
  test/
    happy/
      ...
    sad/
      ...
```

## 3. Face Detection Example

In [ ]:
# Initialize face detector
detector = FaceDetector()

# Example: Load and detect face in an image
# Replace with your actual image path
example_image_path = '../data/train/happy/example.jpg'

if os.path.exists(example_image_path):
    image = cv2.imread(example_image_path)
    face = detector.detect_face(image)
    
    if face is not None:
        # Display original and detected face
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Detected Face')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No face detected in the image")
else:
    print(f"Image not found: {example_image_path}")
    print("Please add your images to the data directory first")

## 4. Create and Visualize CNN Model

In [ ]:
# Create a basic CNN model (simpler architecture for learning)
model = create_basic_cnn(input_shape=(48, 48, 1), num_classes=3)
model = compile_model(model, learning_rate=0.001)

# Print model summary
print_model_summary(model)

## 5. Understanding CNN Layers

### Key Components:

1. **Convolutional Layers (Conv2D)**:
   - Extract features from images
   - Learn patterns like edges, shapes, facial features
   - Filters slide over the image to detect patterns

2. **Pooling Layers (MaxPooling2D)**:
   - Reduce spatial dimensions
   - Keep important features
   - Make model more robust to variations

3. **Flatten Layer**:
   - Convert 2D feature maps to 1D vector
   - Prepare for fully connected layers

4. **Dense Layers**:
   - Fully connected layers
   - Learn complex combinations of features
   - Final layer outputs emotion predictions

5. **Dropout**:
   - Randomly deactivate neurons during training
   - Prevents overfitting
   - Makes model more generalizable

## 6. Prepare Data (if data exists)

In [ ]:
# Check if training data exists
train_dir = '../data/train'
test_dir = '../data/test'

if os.path.exists(train_dir) and len(os.listdir(train_dir)) > 0:
    # Prepare data generators
    train_generator = prepare_dataset_from_directory(
        train_dir,
        target_size=(48, 48),
        batch_size=16,
        augment=True
    )
    
    print(f"Number of training samples: {train_generator.samples}")
    print(f"Number of classes: {len(train_generator.class_indices)}")
    print(f"Class names: {list(train_generator.class_indices.keys())}")
    
    # Visualize some training images
    images, labels = next(train_generator)
    
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.ravel()
    
    class_names = list(train_generator.class_indices.keys())
    
    for i in range(min(8, len(images))):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        label_idx = np.argmax(labels[i])
        axes[i].set_title(class_names[label_idx])
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No training data found.")
    print("Please organize your images in the data/train directory.")

## 7. Train the Model

**Note**: Training is better done using the `train.py` script:

```bash
python train.py --train_dir data/train --test_dir data/test --epochs 30 --model_type basic
```

This allows for better monitoring and checkpoint saving.

In [ ]:
# Example training code (uncomment to train)
# if os.path.exists(train_dir) and len(os.listdir(train_dir)) > 0:
#     history = model.fit(
#         train_generator,
#         epochs=10,
#         verbose=1
#     )
#     
#     # Plot training history
#     plt.figure(figsize=(12, 4))
#     
#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'])
#     plt.title('Model Accuracy')
#     plt.ylabel('Accuracy')
#     plt.xlabel('Epoch')
#     plt.legend(['Train'], loc='upper left')
#     
#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'])
#     plt.title('Model Loss')
#     plt.ylabel('Loss')
#     plt.xlabel('Epoch')
#     plt.legend(['Train'], loc='upper left')
#     
#     plt.tight_layout()
#     plt.show()
#     
#     # Save model
#     model.save('../models/emotion_recognition_model.h5')
#     print("Model saved!")

## 8. Make Predictions (after training)

In [ ]:
# Load trained model (if exists)
model_path = '../models/emotion_recognition_model.h5'

if os.path.exists(model_path):
    # Initialize predictor
    predictor = EmotionPredictor(
        model_path,
        emotion_labels=['happy', 'neutral', 'sad']  # Adjust based on your classes
    )
    
    # Example: Predict on a test image
    test_image_path = '../data/test/happy/example.jpg'
    
    if os.path.exists(test_image_path):
        emotion, confidence, probabilities = predictor.predict_from_file(test_image_path)
        
        # Display image with prediction
        image = cv2.imread(test_image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(image_rgb)
        plt.title(f'Predicted: {emotion.upper()} ({confidence*100:.1f}%)')
        plt.axis('off')
        plt.show()
        
        # Show prediction probabilities
        plt.figure(figsize=(8, 4))
        plt.bar(predictor.emotion_labels, probabilities)
        plt.title('Emotion Probabilities')
        plt.ylabel('Probability')
        plt.xlabel('Emotion')
        plt.ylim([0, 1])
        plt.show()
    else:
        print(f"Test image not found: {test_image_path}")
else:
    print("Model not found. Please train the model first using train.py")

## 9. Key Learning Points

### CNN Architecture:
- **Convolutional layers** learn hierarchical features
- **Early layers** detect simple patterns (edges, textures)
- **Deeper layers** detect complex patterns (facial features, expressions)

### Training Process:
- **Forward pass**: Input flows through network
- **Loss calculation**: Compare predictions with true labels
- **Backpropagation**: Update weights to minimize loss
- **Optimization**: Adam optimizer adjusts learning

### Data Augmentation:
- Helps model generalize better
- Creates variations of training images
- Prevents overfitting to specific poses/lighting

### Important Hyperparameters:
- **Learning rate**: Controls weight update size
- **Batch size**: Number of samples per update
- **Epochs**: Number of complete passes through data
- **Dropout**: Regularization to prevent overfitting

## 10. Next Steps

1. **Collect more data**: Take photos with different emotions
2. **Organize data**: Place in data/train and data/test folders
3. **Train model**: Run `python train.py`
4. **Test predictions**: Run `python predict.py --image path/to/image.jpg`
5. **Try real-time**: Run `python predict.py --webcam`
6. **Experiment**: Try different architectures, hyperparameters